In [0]:
%sql
WITH revenue_cte AS (
    SELECT 
        posting_year,
        posting_month,
        posting_year_month,
        SUM(credit_amount) AS revenue
    FROM 03_global_tech_gold.data_cube.gl_data_cube
    WHERE account_category = 'Revenue'
    GROUP BY posting_year, posting_month, posting_year_month
),
lagged AS (
    SELECT *,
        LAG(revenue) OVER (ORDER BY posting_year, posting_month) AS prev_revenue
    FROM revenue_cte
)
SELECT 
    posting_year_month,
    revenue,
    prev_revenue,
    ROUND((revenue - prev_revenue) / NULLIF(prev_revenue, 0) * 100, 2) AS mom_growth_pct
FROM lagged
ORDER BY posting_year, posting_month;

In [0]:
%sql
SELECT 
    posting_year,
    posting_month,
    posting_year_month,
    company_name,
    SUM(net_amount) AS cost_of_sales
FROM 03_global_tech_gold.data_cube.gl_data_cube
WHERE account_category = 'Cost of Sales'
GROUP BY posting_year, posting_month, posting_year_month, company_name
ORDER BY posting_year, posting_month, company_name;

In [0]:
%sql
WITH revenue AS (
    SELECT 
        posting_year,
        posting_month,
        posting_year_month,
        SUM(net_amount) AS revenue
    FROM 03_global_tech_gold.data_cube.gl_data_cube
    WHERE account_category = 'Revenue'
    GROUP BY posting_year, posting_month, posting_year_month
),
cogs AS (
    SELECT 
        posting_year,
        posting_month,
        posting_year_month,
        SUM(net_amount) AS cogs
    FROM 03_global_tech_gold.data_cube.gl_data_cube
    WHERE account_category = 'Cost of Sales'
    GROUP BY posting_year, posting_month, posting_year_month
)
SELECT 
    r.posting_year_month,
    r.revenue,
    c.cogs,
    (r.revenue - c.cogs) AS gross_profit,
    ROUND((r.revenue - c.cogs) / NULLIF(r.revenue, 0) * 100, 2) AS gross_margin_pct
FROM revenue r
JOIN cogs c ON r.posting_year_month = c.posting_year_month
ORDER BY r.posting_year, r.posting_month;

In [0]:
%sql
SELECT 
    company_name,
    department_name,
    account_name,
    account_category,
    ROUND(SUM(net_amount), 2) AS total_expense
FROM 03_global_tech_gold.data_cube.gl_data_cube
WHERE account_category = 'Operating Expense'
GROUP BY company_name, department_name, account_name, account_category
ORDER BY total_expense DESC;

In [0]:
%sql
WITH revenue AS (
    SELECT 
        posting_year_month,
        posting_year,
        posting_month,
        SUM(net_amount) AS revenue
    FROM 03_global_tech_gold.data_cube.gl_data_cube
    WHERE account_category = 'Revenue'
    GROUP BY posting_year_month, posting_year, posting_month
),
cogs AS (
    SELECT 
        posting_year_month,
        SUM(net_amount) AS cogs
    FROM 03_global_tech_gold.data_cube.gl_data_cube
    WHERE account_category = 'Cost of Sales'
    GROUP BY posting_year_month
),
opex AS (
    SELECT 
        posting_year_month,
        SUM(net_amount) AS opex
    FROM 03_global_tech_gold.data_cube.gl_data_cube
    WHERE account_category = 'Operating Expense'
    GROUP BY posting_year_month
)
SELECT 
    r.posting_year_month,
    r.revenue,
    COALESCE(c.cogs, 0) AS cogs,
    COALESCE(o.opex, 0) AS opex,
    r.revenue - COALESCE(c.cogs, 0) - COALESCE(o.opex, 0) AS net_profit
FROM revenue r
LEFT JOIN cogs c ON r.posting_year_month = c.posting_year_month
LEFT JOIN opex o ON r.posting_year_month = o.posting_year_month
ORDER BY r.posting_year, r.posting_month;

In [0]:
%sql
SELECT 
    position,
    company_name,
    ROUND(AVG(total_compensation),2) AS avg_total_compensation,
    ROUND(AVG(bonus),2) AS avg_bonus,
    ROUND(AVG(overtime_pay),2) AS avg_overtime_pay,
    ROUND(AVG(commission),2) AS avg_commission,
    COUNT(DISTINCT employee_id) AS employee_count
FROM 03_global_tech_gold.data_cube.payroll_data_cube
GROUP BY position, company_name
ORDER BY avg_total_compensation DESC;

In [0]:
%sql
SELECT 
    department_name,
    ROUND(SUM(overtime_pay),2) AS total_overtime,
    ROUND(SUM(bonus),2) AS total_bonus,
    ROUND(SUM(commission),2) AS total_commission,
    ROUND(SUM(overtime_pay) / NULLIF(SUM(total_compensation - overtime_pay - bonus - commission), 0) * 100, 2) AS overtime_to_base_ratio_pct,
    COUNT(DISTINCT employee_id) AS employee_count
FROM 03_global_tech_gold.data_cube.payroll_data_cube
GROUP BY department_name
ORDER BY total_overtime DESC;

In [0]:
%sql
WITH monthly_revenue AS (
    SELECT 
        posting_year_month,
        company_name,
        SUM(net_amount) AS revenue
    FROM 03_global_tech_gold.data_cube.gl_data_cube
    WHERE account_category = 'Revenue'
    GROUP BY posting_year_month, company_name
),
monthly_payroll AS (
    SELECT 
        pay_year_month,
        company_name,
        SUM(total_compensation) AS payroll_cost
    FROM 03_global_tech_gold.data_cube.payroll_data_cube
    GROUP BY pay_year_month, company_name
)
SELECT 
    r.posting_year_month,
    r.company_name,
    r.revenue,
    COALESCE(p.payroll_cost, 0) AS payroll_cost,
    ROUND(COALESCE(p.payroll_cost, 0) / NULLIF(r.revenue, 0) * 100, 2) AS payroll_pct_of_revenue
FROM monthly_revenue r
LEFT JOIN monthly_payroll p 
    ON r.posting_year_month = p.pay_year_month 
    AND r.company_name = p.company_name
ORDER BY r.posting_year_month, r.company_name;

In [0]:
%sql
SELECT 
    department_name,
    company_name,
    ROUND(SUM(total_compensation),2) AS total_payroll_cost,
    COUNT(DISTINCT employee_id) AS employee_count,
    ROUND(AVG(total_compensation), 2) AS avg_cost_per_employee
FROM 03_global_tech_gold.data_cube.payroll_data_cube
GROUP BY department_name, company_name
ORDER BY total_payroll_cost DESC;

In [0]:
%sql
SELECT 
    department_name,
    pay_year_month,
    COUNT(DISTINCT employee_id) AS active_headcount,
    company_name
FROM 03_global_tech_gold.data_cube.payroll_data_cube
WHERE employee_is_active = true
GROUP BY department_name, pay_year_month, company_name
ORDER BY pay_year_month, department_name;